(tutorial_advanced_debugging_label)=

# Debugging the solution process

The {ref}`debugging tutorial <tutorial_debugging_label>` covers specification
errors on a small model. This tutorial goes one step further: it walks through
the solution process of a larger model and shows how to use the solver's
inspection tools when the solve fails.

1. how the problem reduction, the block decomposition and the starting values
   of a successful solve can be inspected,
2. how structural over- and underdetermination is reported and what a
   structurally singular but count-balanced problem looks like,
3. how the specifications shape the block decomposition,
4. how to run the solution process in stages with :code:`presolve` and
   :code:`solve_continue` and
5. how to pause the solver at a failed block, inspect the failure and decide
   whether the starting values or the specification are the problem.

Background information on all steps of the solution process is available in
the {ref}`solver section <modules_solver_label>`. The outputs shown here are
based on the following version of tespy:

In [ ]:
from tespy import __version__
__version__

## The example model

The model is an Organic Rankine Cycle with an internal recuperator, an
air cooled condenser and a geothermal heat source, the same plant as in the
{ref}`workflow integration section <integration_model_class_template_label>`.

```{figure} /_static/images/tutorials/orc/flowsheet.svg
:align: center
:class: only-light
:width: 80%

Flowsheet of the recuperated ORC system
```

```{figure} /_static/images/tutorials/orc/flowsheet_darkmode.svg
:align: center
:class: only-dark
:width: 80%

Flowsheet of the recuperated ORC system
```

The working fluid is isopentane. The evaporation level is fixed through
saturated turbine inlet at 150 °C, the condensation level through the air
temperatures and the pinch specifications of the condenser.

```{note}
Since the following
sections modify the model repeatedly, the setup lives in a function returning a
freshly parametrized network.
```

In [ ]:
from tespy.components import (
    CycleCloser, Generator, Motor, MovingBoundaryHeatExchanger,
    PowerBus, PowerSink, Pump, Sink, Source, Turbine,
)
from tespy.connections import Connection, PowerConnection
from tespy.networks import Network


def build_orc():
    nw = Network(iterinfo=False)
    nw.units.set_defaults(
        temperature="degC", pressure="bar", pressure_difference="bar",
        power="kW", heat="kW"
    )

    turbine = Turbine("turbine")
    recuperator = MovingBoundaryHeatExchanger("recuperator")
    condenser = MovingBoundaryHeatExchanger("condenser")
    pump = Pump("pump")
    preheater = MovingBoundaryHeatExchanger("preheater")
    evaporator = MovingBoundaryHeatExchanger("evaporator")
    cc = CycleCloser("cc")

    heat_source = Source("heat source")
    heat_outflow = Sink("heat outflow")
    air_source = Source("air source")
    air_sink = Sink("air sink")

    a1 = Connection(heat_source, "out1", evaporator, "in1", label="a1")
    a2 = Connection(evaporator, "out1", preheater, "in1", label="a2")
    a3 = Connection(preheater, "out1", heat_outflow, "in1", label="a3")

    b1 = Connection(cc, "out1", turbine, "in1", label="b1")
    b2 = Connection(turbine, "out1", recuperator, "in1", label="b2")
    b3 = Connection(recuperator, "out1", condenser, "in1", label="b3")
    b4 = Connection(condenser, "out1", pump, "in1", label="b4")
    b5 = Connection(pump, "out1", recuperator, "in2", label="b5")
    b6 = Connection(recuperator, "out2", preheater, "in2", label="b6")
    b7 = Connection(preheater, "out2", evaporator, "in2", label="b7")
    b8 = Connection(evaporator, "out2", cc, "in1", label="b8")

    c1 = Connection(air_source, "out1", condenser, "in2", label="c1")
    c2 = Connection(condenser, "out2", air_sink, "in1", label="c2")

    nw.add_conns(a1, a2, a3, b1, b2, b3, b4, b5, b6, b7, b8, c1, c2)

    generator = Generator("generator")
    motor = Motor("motor")
    power_bus = PowerBus("bus", num_in=1, num_out=2)
    grid = PowerSink("grid")

    e1 = PowerConnection(turbine, "power", generator, "power_in", label="e1")
    e2 = PowerConnection(generator, "power_out", power_bus, "power_in1", label="e2")
    e3 = PowerConnection(power_bus, "power_out1", motor, "power_in", label="e3")
    e4 = PowerConnection(motor, "power_out", pump, "power", label="e4")
    e5 = PowerConnection(power_bus, "power_out2", grid, "power", label="e5")

    nw.add_conns(e1, e2, e3, e4, e5)

    generator.set_attr(eta=0.98)
    motor.set_attr(eta=0.98)

    a1.set_attr(fluid={"air": 1}, T=200, p=1, m=10)
    b1.set_attr(fluid={"Isopentane": 1}, x=1, T=150)
    b3.set_attr(td_dew=10)
    b4.set_attr(td_bubble=5)
    b7.set_attr(td_bubble=5)
    c1.set_attr(fluid={"air": 1}, T=10, p=1)
    c2.set_attr(T=20)

    recuperator.set_attr(dp1=0, dp2=0)
    condenser.set_attr(dp1=0, dp2=0)
    preheater.set_attr(dp1=0, dp2=0)
    evaporator.set_attr(dp1=0, dp2=0)

    turbine.set_attr(eta_s=0.8)
    pump.set_attr(eta_s=0.7)

    condenser.set_attr(td_pinch=5)
    evaporator.set_attr(td_pinch=10)
    return nw

The model is well specified and solves without any issues:

In [ ]:
nw = build_orc()
nw.solve("design")
nw.assert_convergence()
print(f"net power: {nw.get_conn('e5').E.val:.1f} kW")

## Inspecting a successful solve

Before breaking the model, we look at what the solver does with it when
everything works. All inspection methods are available after
{py:meth}`~tespy.networks.network.Network.presolve`, which prepares and
presolves the problem but stops before solving.

The original problem holds 57 variables. mass flow, pressure and enthalpy
per :code:`Connection` and an energy flow per :code:`PowerConnection`. The
reduction merges linearly coupled variables (e.g. all mass flows of the
cycle, the pressures connected through the :code:`dp` specifications) and
presolves everything the specifications determine directly. 13 variables
remain:

In [ ]:
nw = build_orc()
nw.presolve("design")
nw.print_variables()

Each remaining variable has a short label (e.g. :code:`h4`) and lists the
original variables it represents. The same reduction applies to the equations.

- :code:`nw.print_equations()`,
- :code:`nw.print_presolved_variables()` and
- :code:`nw.print_presolved_equations()` show the tables.

Next, the solver analyses which equation determines which variable and in
what order the equations can be processed, see
{ref}`the decomposition section <solver_decomposition_label>` for how this
works:

In [ ]:
nw.print_blocks()

The decomposition allows to analyze the problem on the mathematical level. You
will find that only the three equations fix the condensation state, i.e. the
condenser pinch together with the two saturation offsets. Everything else
follows sequentially: The isentropic efficiencies determine the machine outlet
enthalpies, the energy balances determine the mass flows and the power
connector balances determine the energy flows. The :code:`Needs` column lists
which blocks must be solved before solving a block.

The block lower triangular structure can also be displayed as an incidence
matrix, with :code:`X` marking the entries of an equation within its own
block and :code:`x` marking dependencies on variables of preceding blocks:

In [ ]:
nw.print_incidence_matrix(block_order=True)

The starting values generated for the remaining variables are inspected
with :code:`print_variable_values`. All values are SI values:

In [ ]:
nw.print_variable_values()

`solve_continue` runs the solution process on the prepared problem
and finishes the calculation:

In [ ]:
nw.solve_continue()
print(nw.status)

## Structural errors

The determination check compares the number of unknowns with the number of
equations. With the maximum matching of the
{ref}`decomposition <solver_decomposition_label>` the solver additionally
locates the part of the problem in which a defect sits. Specifying the
evaporator heat transfer on top of the well determined model will show what
equations overdetermine which variable(s).

In [ ]:
nw = build_orc()
nw.get_comp("evaporator").set_attr(Q=-1e6)
nw.solve("design")

The structural analysis remains available after the error is raised and you can
print the affected equations and variables:

In [ ]:
nw.print_structural_analysis()

The evaporator heat transfer and its pinch specification both determine the
enthalpy at the evaporator air outlet (:code:`h1`).

Similarly, if we remove a specification from a fresh network instance, we
produce an under-determined part:

In [ ]:
nw = build_orc()
nw.get_conn("c2").set_attr(T=None)
nw.solve("design")

In [ ]:
nw.print_structural_analysis()

The list is longer here because the missing equation propagates through
multiple parts. without the air outlet temperature the condenser energy balance
cannot determine the air mass flow, without the air mass flow the condensation
level is open, and so on.

A more subtle case is a **count-balanced but structurally singular**
problem. If we remove the air outlet temperature and specify the evaporator
heat transfer, we will see 14 equations for 14 variables. However, now the
evaporator side is over-determined and the condenser side is under-determined
at the same time. Block-wise solving is impossible in this case and the solver
reports the structural analysis and exits with status 3. 

In [ ]:
nw = build_orc()
nw.get_conn("c2").set_attr(T=None)
nw.get_comp("evaporator").set_attr(Q=-1e6)
nw.solve("design")
print(nw.status)

Both defective parts are visible at once here:

In [ ]:
nw.print_structural_analysis()

```{tip}
The message also points to :code:`block_solve=False` to attempt the
simultaneous solution anyway, for the case that the structural defect stems
from an incomplete dependency declaration of a custom component rather than
from the specifications.
```

The structural analysis works on the incidence only (which equation depends on
which variable). A linear dependency can also arise from the values even when
the structure is sound, e.g. redundant specifications connected through
nonlinear relations or starting values on a saturation plateau. The solver then
reports status 3 and the singularity diagnosis names the suspect equations in
the log, see {ref}`the debugging section <networks_debugging_label>`.

## Specifications and decomposition

Changing a specification changes the block structure. In the base model the
mass flow of the heat source is given and every block is determined in
sequence. If we instead prescribe the net power of the plant, the mass flows
can only be determined together with the energy flows through the bus balance:

In [ ]:
nw = build_orc()
nw.get_conn("a1").set_attr(m=None)
nw.get_conn("e5").set_attr(E=180)
nw.presolve("design")
nw.print_blocks()

Block 4 now couples the cycle mass flow with the turbine and pump energy flows.
The bus balance connects generator output and motor input, which are tied to
the same mass flow through the connector balances. The larger the blocks of
your model the harder will debugging be. However, especially in offdesign
simulations tightly coupled problems that cannot easily be reduced to multiple
subproblems are expected.

## Staged solving

`solve` first executes `presolve` and subsequently `solve_continue`. You can
pause the model execution between the two stages and inspect the problem, the
variables and their values and even modify the initial guesses yourself (values
are SI values). On this model the generated starting values are good enough
that staging is not needed, therefore the cell only intends to show how the
mechanism works.

In [ ]:
nw = build_orc()
nw.presolve("design")
nw.set_variable_value("b4", "p", 1.2e5)
nw.solve_continue()
print(nw.status)

## Pausing and inspecting a failed block

This section will show how you can set starting values selectively for a
failing block. To construct the example, we first run the simulation with a
tiny pinch point value for the condenser, retrieve the resulting UA and impose
it to a freshly constructed model. Then we see that the new solve will fail
with a non-convergence warning.

In [ ]:
nw = build_orc()
nw.get_comp("condenser").set_attr(td_pinch=0.001)
nw.solve("design")
nw.assert_convergence()
UA_value = nw.get_comp("condenser").UA.val

In [ ]:
nw.results["Connection"].loc[["c1", "b3", "b4"], ["m", "p", "h"]]

In [ ]:
nw = build_orc()
nw.get_comp("condenser").set_attr(td_pinch=None, UA=UA_value)
nw.solve("design", pause_on_block_failure=True)
nw.status

We can inspect the fluid state at failure, the variables relevant to the block
are marked with a star.

In [ ]:
nw.print_block_states(block=2, at="failure")

And we can inspect the same before starting the block solve. Simultaneously, we
can also inspect the variables directly when starting the block solve.

In [ ]:
nw.print_block_states(block=2)
nw.print_variable_values(block=2)

We can transfer the variable values from the tiny pinch solution and continue
solving. In this case we apply `osciallation_damping` to help the convergence,
since a tiny change in any of the variables will trigger a sign change in the
`UA` residual.

In [ ]:
nw.set_variable_value("b3", "h", 347635)
nw.set_variable_value("c1", "m", 78)
nw.set_variable_value("b4", "h", -29941)
nw.set_variable_value("b3", "p", 754.166)
nw.solve_continue()
nw.status

In case you want to continue solving without pausing on a failure, you can run
`nw.solve_continue(pause_on_block_failure=False)`.